<a href="https://colab.research.google.com/github/lherwehe/14ers-git/blob/master/lc_maps_spacetime_graphics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Setup
# Install deps (quiet)
!pip -q install pandas geopandas gcsfs fsspec rasterio mapclassify seaborn imageio libpysal esda shapely pyproj contextily

import os, sys, subprocess, io, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from textwrap import wrap

# Optional (rasters, GIFs)
try:
    import rasterio, rasterio.plot
    import imageio
    RASTER_OK = True
except Exception:
    RASTER_OK = False

# Spatial autocorrelation
from libpysal.weights import Queen
from esda.moran import Moran

sns.set_context("talk")
sns.set_style("whitegrid")

from google.colab import auth
auth.authenticate_user()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.4 MB/s eta 0:00:00


In [3]:
# --- Config (edit as needed) ---
INPUT_CSV   = 'gs://moz-lulc/MOZ_cultivated_change_admin3_fast.csv'  # latest CSV from GEE
ADMIN_GEO   = 'gs://moz-lulc/shp/moz_dist/moz_admbnda_adm3_ine_20190607.shp'  # admin-post boundaries

OUTPUT_DIR_GCS = 'gs://moz-lulc/figs'  # where to copy figures in GCS
LOCAL_OUTDIR   = '/content/figs'       # local staging dir for figures

# Provinces of interest (match GEE)
PROVINCES = ['Cabo Delgado', 'Nampula', 'Sofala', 'Gaza']

# CSV fields from GEE
CSV_PROV_FIELD = "province"
CSV_AP_FIELD   = "admin_post"

# Geo fields
GEO_PROV_FIELD = "ADM1_PT"
GEO_AP_FIELD   = "ADM3_PT"

# Optional rasters (downloaded on-demand from GCS)
RASTER_LOCAL_DIR = "/content/rasters"
RASTER_GCS_PREFIXES = {
    "baseline": "gs://moz-lulc/MOZ_cultivated_change_baseline_prob_2018_2020_",
    "endline":  "gs://moz-lulc/MOZ_cultivated_change_endline_prob_2022_2024_",
    "delta":    "gs://moz-lulc/MOZ_cultivated_change_delta_prob_end_minus_base_",
}

os.makedirs(LOCAL_OUTDIR, exist_ok=True)
os.makedirs(RASTER_LOCAL_DIR, exist_ok=True)

# --- Helpers for GCS IO ---
def is_gcs(path: str) -> bool:
    return str(path).startswith("gs://")

def gcs_cp(src: str, dst: str, recursive: bool=False) -> None:
    cmd = ["gsutil", "-m", "cp"]
    if recursive:
        cmd.append("-r")
    cmd += [src, dst]
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0:
        print(r.stdout)
        raise RuntimeError(f"gsutil cp failed: {src} -> {dst}")

def savefig(fname_local: str, tight: bool=True, dpi: int=200, to_gcs: str|None=None):
    if tight:
        plt.tight_layout()
    plt.savefig(fname_local, dpi=dpi, bbox_inches="tight")
    print(f"Saved local: {fname_local}")
    if to_gcs and is_gcs(to_gcs):
        gcs_cp(fname_local, to_gcs)

def gcs_read_csv(path: str) -> pd.DataFrame:
    # pandas + gcsfs handles gs:// directly
    return pd.read_csv(path)

def fetch_shapefile_from_gcs(gcs_shp_path: str, local_dir: str) -> str:
    # Download all sidecar files using wildcard on base name
    base_no_ext = os.path.splitext(gcs_shp_path)[0]
    os.makedirs(local_dir, exist_ok=True)
    gcs_cp(base_no_ext + ".*", local_dir)
    local_shp = os.path.join(local_dir, os.path.basename(base_no_ext) + ".shp")
    if not os.path.exists(local_shp):
        raise FileNotFoundError(f"Local .shp not found at {local_shp}")
    return local_shp

def fetch_raster_from_gcs(gcs_prefix: str, province: str) -> str|None:
    # Compose full GCS path by appending province tokenified + ".tif"
    stem = f"{gcs_prefix}{province.replace(' ','_')}.tif"
    local_path = os.path.join(RASTER_LOCAL_DIR, os.path.basename(stem))
    try:
        gcs_cp(stem, local_path)
        return local_path
    except Exception as e:
        print(f"Raster missing for {province}: {stem} ({e})")
        return None

# --- Load CSV ---
df = gcs_read_csv(INPUT_CSV)

# Clean join keys
for col in [CSV_PROV_FIELD, CSV_AP_FIELD]:
    df[col] = df[col].astype(str).str.strip()

# Filter to desired provinces
df = df[df[CSV_PROV_FIELD].isin(PROVINCES)].copy()

# Expected core columns (uid optional)
core_cols = [
    "ha_gain","ha_loss","ha_stable_cult","ha_stable_uncult",
    "ha_prob_baseline","ha_prob_endline","ha_prob_change","ha_wcereal21"
]

# Optional uncertainty columns (from updated GEE export)
var_cols = ["var_prob_baseline_ha2","var_prob_endline_ha2","var_prob_change_ha2"]
se_cols  = ["se_prob_baseline_ha","se_prob_endline_ha","se_prob_change_ha"]

missing_core = [c for c in core_cols if c not in df.columns]
if missing_core:
    raise ValueError(f"Missing expected columns in CSV: {missing_core}")

# Optional uid: create if absent
if "uid" not in df.columns:
    df["uid"] = np.arange(1, len(df) + 1)

# Ensure numeric only for columns that exist
num_cols = [c for c in (core_cols + var_cols + se_cols) if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# If SEs missing but variances present, derive SE
for v, s in zip(var_cols, se_cols):
    if (v in df.columns) and (s not in df.columns):
        df[s] = np.sqrt(df[v].clip(lower=0.0))

# If both missing, set SEs to zero
for s in se_cols:
    if s not in df.columns:
        df[s] = 0.0

# Derive 95% CIs
df["ci95_lo_base"] = (df["ha_prob_baseline"] - 1.96 * df["se_prob_baseline_ha"]).clip(lower=0.0)
df["ci95_hi_base"] = df["ha_prob_baseline"] + 1.96 * df["se_prob_baseline_ha"]
df["ci95_lo_end"]  = (df["ha_prob_endline"] - 1.96 * df["se_prob_endline_ha"]).clip(lower=0.0)
df["ci95_hi_end"]  = df["ha_prob_endline"] + 1.96 * df["se_prob_endline_ha"]
df["ci95_lo_chg"]  = df["ha_prob_change"] - 1.96 * df["se_prob_change_ha"]
df["ci95_hi_chg"]  = df["ha_prob_change"] + 1.96 * df["se_prob_change_ha"]

# --- Figure 1: Province-level stacked bars (stable, gain, loss) ---
prov_sum = df.groupby(CSV_PROV_FIELD, as_index=False)[["ha_stable_cult","ha_gain","ha_loss"]].sum()
prov_sum["total"] = prov_sum["ha_stable_cult"] + prov_sum["ha_gain"] + prov_sum["ha_loss"]
prov_sum = prov_sum.sort_values("total", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(prov_sum[CSV_PROV_FIELD], prov_sum["ha_stable_cult"], label="Stable cultivated", color="#2ca25f")
ax.bar(prov_sum[CSV_PROV_FIELD], prov_sum["ha_gain"], bottom=prov_sum["ha_stable_cult"], label="Gain", color="#3182bd")
ax.bar(prov_sum[CSV_PROV_FIELD],
       prov_sum["ha_loss"],
       bottom=prov_sum["ha_stable_cult"] + prov_sum["ha_gain"],
       label="Loss", color="#de2d26")
ax.set_ylabel("Hectares"); ax.set_xlabel("Province")
ax.set_title("Cultivated area components by province (ha)")
ax.legend(ncol=3, frameon=True); plt.xticks(rotation=15)
savefig(os.path.join(LOCAL_OUTDIR, "stacked_bars_province_stable_gain_loss.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "stacked_bars_province_stable_gain_loss.png"))
plt.close()

# --- Figure 2: Violin plots of ha_prob_change by province ---
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=df, x=CSV_PROV_FIELD, y="ha_prob_change", inner="quartile", palette="Set2", ax=ax, cut=0)
ax.set_xlabel("Province"); ax.set_ylabel("Change in cultivated (ha), 2022–2024 minus 2018–2020")
ax.set_title("Distribution of cultivated change by admin post"); plt.xticks(rotation=15)
savefig(os.path.join(LOCAL_OUTDIR, "violin_change_by_province.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "violin_change_by_province.png"))
plt.close()

# --- Figure 3: ECDF plots of ha_prob_change by province ---
fig, ax = plt.subplots(figsize=(8, 6))
for p, sub in df.groupby(CSV_PROV_FIELD):
    x = np.sort(sub["ha_prob_change"].values)
    y = np.arange(1, len(x) + 1) / len(x)
    ax.plot(x, y, label=p)
ax.set_xlabel("Change in cultivated (ha)"); ax.set_ylabel("ECDF")
ax.set_title("ECDF of cultivated change by province"); ax.legend(frameon=True)
savefig(os.path.join(LOCAL_OUTDIR, "ecdf_change_by_province.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "ecdf_change_by_province.png"))
plt.close()

# --- Figure 4: Rank-order bars of ha_prob_change by admin post (faceted by province) ---
ncols = 2
nrows = int(np.ceil(len(PROVINCES) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 4.5 * nrows), sharex=False)
axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
for i, p in enumerate(PROVINCES):
    ax = axes[i]
    sub = df[df[CSV_PROV_FIELD] == p].copy().sort_values("ha_prob_change", ascending=False)
    colors = np.where(sub["ha_prob_change"]>=0, "#3182bd", "#de2d26")
    ax.barh(sub[CSV_AP_FIELD], sub["ha_prob_change"], color=colors)
    ax.set_title(p); ax.invert_yaxis(); ax.set_xlabel("Change in cultivated (ha)")
    ax.tick_params(axis='y', labelsize=8)
for j in range(i+1, len(axes)):
    axes[j].axis("off")
plt.suptitle("Ranked cultivated change by admin post (within province)", y=1.0, fontsize=14)
plt.tight_layout()
savefig(os.path.join(LOCAL_OUTDIR, "ranked_change_by_adminpost_faceted.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "ranked_change_by_adminpost_faceted.png"))
plt.close()

# --- Load boundaries and join ---
local_shp = fetch_shapefile_from_gcs(ADMIN_GEO, "/content/admin_geo") if is_gcs(ADMIN_GEO) else ADMIN_GEO
gdf = gpd.read_file(local_shp)

# Normalize join keys
gdf[GEO_PROV_FIELD] = gdf[GEO_PROV_FIELD].astype(str).str.strip()
gdf[GEO_AP_FIELD]   = gdf[GEO_AP_FIELD].astype(str).str.strip()

# Prepare join table with rename to match geo fields
join_df = df.rename(columns={CSV_PROV_FIELD: GEO_PROV_FIELD, CSV_AP_FIELD: GEO_AP_FIELD})
gdf = gdf.merge(join_df, on=[GEO_PROV_FIELD, GEO_AP_FIELD], how="left", validate="1:1")

print("Unmatched polygons:", gdf["ha_prob_change"].isna().sum())
fill_cols = ["ha_prob_change","ha_prob_baseline","ha_prob_endline",
             "se_prob_baseline_ha","se_prob_endline_ha","se_prob_change_ha"]
for c in fill_cols:
    if c in gdf.columns:
        gdf[c] = gdf[c].fillna(0.0)

# CRS (optional)
try:
    gdf = gdf.to_crs(epsg=3857)
except Exception:
    pass

# --- Figure 5: Choropleths for baseline, endline, change ---
def plot_choropleth(gdf_in, column, title, cmap="viridis", center=None, fname="map.png"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 9))
    if center is None:
        gdf_in.plot(column=column, cmap=cmap, linewidth=0.2, edgecolor="gray", legend=True, ax=ax, scheme="Quantiles", k=5)
    else:
        v = gdf_in[column].to_numpy()
        v = v[np.isfinite(v)]
        vmin, vmax = np.nanpercentile(v, 2), np.nanpercentile(v, 98)
        vmax = max(abs(vmin), abs(vmax)); vmin = -vmax
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        gdf_in.plot(column=column, cmap=cmap, linewidth=0.2, edgecolor="gray", legend=False, ax=ax, norm=norm)
        cbar = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        cb = fig.colorbar(cbar, ax=ax, shrink=0.6); cb.set_label("Hectares")
    ax.set_axis_off(); ax.set_title("\n".join(wrap(title, 60)))
    savefig(os.path.join(LOCAL_OUTDIR, fname), to_gcs=os.path.join(OUTPUT_DIR_GCS, fname))
    plt.close()

plot_choropleth(gdf, "ha_prob_baseline", "Baseline cultivated (expected ha), 2018–2020", "YlGn", None, "map_baseline.png")
plot_choropleth(gdf, "ha_prob_endline",  "Endline cultivated (expected ha), 2022–2024", "YlGn", None, "map_endline.png")
plot_choropleth(gdf, "ha_prob_change",   "Change in cultivated (ha), end minus base", "RdBu_r", center=0.0, fname="map_change_diverging.png")

# --- Figure 6: Small-multiples of change by province ---
for p in PROVINCES:
    sub = gdf[gdf[GEO_PROV_FIELD] == p]
    if len(sub) == 0: continue
    fname = f"map_change_{p.replace(' ','_')}.png"
    plot_choropleth(sub, "ha_prob_change", f"Change in cultivated (ha), {p}", "RdBu_r", center=0.0, fname=fname)

# --- Figure 7: Turnover (dominant class by admin post) ---
def dominant_turnover(row):
    areas = {
        1: row.get("ha_stable_cult", 0),
        2: row.get("ha_gain", 0),
        3: row.get("ha_loss", 0),
        4: row.get("ha_stable_uncult", 0),
    }
    return max(areas, key=areas.get)

gdf["turnover_cls"] = gdf.apply(dominant_turnover, axis=1)
turnover_labels = {1: "Stable cultivated", 2: "Gain", 3: "Loss", 4: "Stable uncultivated"}
turnover_colors = {1: "#2ca25f", 2: "#3182bd", 3: "#de2d26", 4: "#9e9e9e"}
turnover_cmap = ListedColormap([turnover_colors[i] for i in [1,2,3,4]])

fig, ax = plt.subplots(1, 1, figsize=(9, 9))
gdf.plot(column="turnover_cls", categorical=True, cmap=turnover_cmap, linewidth=0.2, edgecolor="gray", legend=False, ax=ax)
handles = [mpl.patches.Patch(color=turnover_colors[i], label=turnover_labels[i]) for i in [1,2,3,4]]
ax.legend(handles=handles, frameon=True, title="Turnover", loc="lower left")
ax.set_axis_off(); ax.set_title("Turnover map (dominant class by admin post)")
savefig(os.path.join(LOCAL_OUTDIR, "map_turnover_dominant.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "map_turnover_dominant.png"))
plt.close()

# --- Figure 8: Bivariate choropleth (baseline vs change) ---
gdf["_base_q"]  = pd.qcut(gdf["ha_prob_baseline"].rank(method="first"), 3, labels=[0,1,2])
gdf["_chg_q"]   = pd.qcut(gdf["ha_prob_change"].rank(method="first"), 3, labels=[0,1,2])
gdf["_bi_class"] = gdf["_base_q"].astype(int) * 3 + gdf["_chg_q"].astype(int)
bi_colors = [
    "#e8e8e8", "#b5c0da", "#6c83b5",
    "#b8d6be", "#90b2b3", "#567994",
    "#73ae80", "#5a9178", "#2a5a5b"
]
bi_cmap = ListedColormap(bi_colors)

fig, ax = plt.subplots(1, 1, figsize=(9, 9))
gdf.plot(column="_bi_class", cmap=bi_cmap, linewidth=0.2, edgecolor="gray", legend=False, ax=ax)
ax.set_axis_off(); ax.set_title("Bivariate: baseline (x) vs change (y)")
from matplotlib.gridspec import GridSpec
gs = GridSpec(3, 3, left=0.72, bottom=0.12, right=0.95, top=0.35, wspace=0.0, hspace=0.0, figure=fig)
for i in range(3):
    for j in range(3):
        axl = fig.add_subplot(gs[2-i, j])
        axl.set_xticks([]); axl.set_yticks([]); axl.set_frame_on(True)
        axl.set_facecolor(bi_colors[i*3 + j])
fig.text(0.71, 0.37, "Change \n(low→high)", fontsize=9, va="bottom")
fig.text(0.72, 0.08, "Baseline (low→high)", fontsize=9)
savefig(os.path.join(LOCAL_OUTDIR, "map_bivariate_baseline_vs_change.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "map_bivariate_baseline_vs_change.png"))
plt.close()

# --- New QC: Province-level means with 95% CI (baseline, endline, change) ---
agg = df.groupby(CSV_PROV_FIELD, as_index=False).agg(
    mean_base=("ha_prob_baseline","mean"),
    mean_end=("ha_prob_endline","mean"),
    mean_chg=("ha_prob_change","mean"),
    se_base=("se_prob_baseline_ha","mean"),
    se_end=("se_prob_endline_ha","mean"),
    se_chg=("se_prob_change_ha","mean"),
)
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, col, se, title in zip(
    axes, ["mean_base","mean_end","mean_chg"], ["se_base","se_end","se_chg"],
    ["Baseline (mean ±1.96·SE)","Endline (mean ±1.96·SE)","Change (mean ±1.96·SE)"]
):
    ax.errorbar(agg[CSV_PROV_FIELD], agg[col], yerr=1.96*agg[se], fmt='o', capsize=4)
    ax.set_title(title); ax.set_xticklabels(agg[CSV_PROV_FIELD], rotation=15); ax.set_ylabel("Hectares")
plt.tight_layout()
savefig(os.path.join(LOCAL_OUTDIR, "province_means_with_ci.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "province_means_with_ci.png"))
plt.close()

# --- New QC: Caterpillar plot (admin posts sorted by change with 95% CI) ---
top_n = 40  # show top N by magnitude
df_ci = df.copy()
df_ci["lo"] = df_ci["ci95_lo_chg"]; df_ci["hi"] = df_ci["ci95_hi_chg"]
df_ci["abs_chg"] = df_ci["ha_prob_change"].abs()
df_top = df_ci.nlargest(top_n, "abs_chg").sort_values("ha_prob_change")
fig, ax = plt.subplots(figsize=(10, max(6, top_n*0.2)))
y = np.arange(len(df_top))
ax.hlines(y, df_top["lo"], df_top["hi"], color="gray", alpha=0.6)
ax.plot(df_top["ha_prob_change"], y, "o", color="#333")
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_yticks(y); ax.set_yticklabels(df_top[CSV_AP_FIELD] + " (" + df_top[CSV_PROV_FIELD] + ")")
ax.set_xlabel("Change in cultivated (ha)"); ax.set_title(f"Top {top_n} changes with 95% CI")
plt.tight_layout()
savefig(os.path.join(LOCAL_OUTDIR, "caterpillar_change_topN.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "caterpillar_change_topN.png"))
plt.close()

# --- New QC: Uncertainty maps (SE and relative SE for change) ---
gdf["rel_se_change"] = (gdf["se_prob_change_ha"] / (gdf["ha_prob_change"].abs() + 1e-6)).replace([np.inf, -np.inf], np.nan).clip(upper=10)
plot_choropleth(gdf, "se_prob_change_ha", "SE of change (ha)", "OrRd", None, "map_se_change.png")
plot_choropleth(gdf, "rel_se_change", "Relative SE of change (SE / |change|)", "PuBuGn", None, "map_relse_change.png")

# --- New QC: End vs Base scatter (hexbin) ---
fig, ax = plt.subplots(figsize=(7,6))
hb = ax.hexbin(df["ha_prob_baseline"], df["ha_prob_endline"], gridsize=40, cmap="viridis", mincnt=1)
ax.plot([df["ha_prob_baseline"].min(), df["ha_prob_baseline"].max()],
        [df["ha_prob_baseline"].min(), df["ha_prob_baseline"].max()], 'r--', lw=1)
ax.set_xlabel("Baseline expected ha"); ax.set_ylabel("Endline expected ha")
ax.set_title("Endline vs Baseline (admin posts)")
cb = plt.colorbar(hb, ax=ax); cb.set_label("Count")
savefig(os.path.join(LOCAL_OUTDIR, "hex_end_vs_base.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "hex_end_vs_base.png"))
plt.close()

# --- New QC: Funnel plots (SE vs mean) ---
fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].scatter(df["ha_prob_baseline"], df["se_prob_baseline_ha"], alpha=0.6)
axes[0].set_xlabel("Baseline expected ha"); axes[0].set_ylabel("SE (ha)"); axes[0].set_title("Funnel: Baseline")
axes[1].scatter(df["ha_prob_change"], df["se_prob_change_ha"], alpha=0.6)
axes[1].set_xlabel("Change expected ha"); axes[1].set_ylabel("SE (ha)"); axes[1].set_title("Funnel: Change")
plt.tight_layout()
savefig(os.path.join(LOCAL_OUTDIR, "funnel_se_vs_mean.png"),
        to_gcs=os.path.join(OUTPUT_DIR_GCS, "funnel_se_vs_mean.png"))
plt.close()

# --- New QC: Global Moran's I for change + Moran scatter ---
# Build contiguity weights on valid polygons
gdf_valid = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
y = gdf_valid["ha_prob_change"].to_numpy()
if np.isfinite(y).sum() >= 5:
    try:
        W = Queen.from_dataframe(gdf_valid)
        W.transform = "r"
        mi = Moran(y, W)
        print(f"Global Moran's I (change): I={mi.I:.3f}, p={mi.p_sim:.4f}, z={mi.z_norm:.2f}")

        z = (y - np.nanmean(y)) / (np.nanstd(y) + 1e-12)
        lag_z = W.sparse.tocsr().dot(z)
        fig, ax = plt.subplots(figsize=(6,6))
        ax.scatter(z, lag_z, alpha=0.6)
        ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)
        # regression line slope approximates Moran's I
        m = np.polyfit(z, lag_z, 1)[0]
        xline = np.linspace(z.min(), z.max(), 100)
        ax.plot(xline, m*xline, 'r--', label=f"Slope≈{m:.2f}")
        ax.set_xlabel("Standardized change"); ax.set_ylabel("Spatial lag of change")
        ax.set_title("Moran scatter (change)"); ax.legend()
        savefig(os.path.join(LOCAL_OUTDIR, "moran_scatter_change.png"),
                to_gcs=os.path.join(OUTPUT_DIR_GCS, "moran_scatter_change.png"))
        plt.close()
    except Exception as e:
        print("Moran computation skipped:", e)
else:
    print("Insufficient valid data for Moran's I")

# --- Optional: Raster visualization + simple animation (if TIFs in GCS) ---
def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        arr = np.where(np.isfinite(arr), arr, np.nan)
        extent = rasterio.plot.plotting_extent(src)
    return arr, extent

def plot_raster(arr, extent, title, cmap="YlGn", fname="raster.png", center=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    if center is None:
        im = ax.imshow(arr, extent=extent, cmap=cmap)
        cb = fig.colorbar(im, ax=ax, shrink=0.7)
    else:
        v = arr[np.isfinite(arr)]
        vmin, vmax = np.nanpercentile(v, 2), np.nanpercentile(v, 98)
        vmax = max(abs(vmin), abs(vmax)); vmin = -vmax
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        im = ax.imshow(arr, extent=extent, cmap=cmap, norm=norm)
        cb = fig.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(title); ax.set_axis_off()
    savefig(os.path.join(LOCAL_OUTDIR, fname),
            to_gcs=os.path.join(OUTPUT_DIR_GCS, fname))
    plt.close()

if RASTER_OK:
    for p in PROVINCES:
        b_gcs = RASTER_GCS_PREFIXES["baseline"]
        e_gcs = RASTER_GCS_PREFIXES["endline"]
        d_gcs = RASTER_GCS_PREFIXES["delta"]
        bpath = fetch_raster_from_gcs(b_gcs, p)
        epath = fetch_raster_from_gcs(e_gcs, p)
        dpath = fetch_raster_from_gcs(d_gcs, p)
        if all([bpath, epath, dpath]):
            b, b_ext = load_raster(bpath)
            e, e_ext = load_raster(epath)
            d, d_ext = load_raster(dpath)
            plot_raster(b, b_ext, f"Baseline probability mean 2018–2020 ({p})", "YlGn", f"raster_baseline_{p.replace(' ','_')}.png")
            plot_raster(e, e_ext, f"Endline probability mean 2022–2024 ({p})", "YlGn", f"raster_endline_{p.replace(' ','_')}.png")
            plot_raster(d, d_ext, f"Delta probability ({p})", "RdBu_r", f"raster_delta_{p.replace(' ','_')}.png", center=0.0)
            # Simple animation (baseline→endline)
            frames = []
            for arr, title in [(b, "2018–2020"), (e, "2022–2024")]:
                fig, ax = plt.subplots(figsize=(6, 5))
                im = ax.imshow(arr, extent=b_ext, cmap="YlGn")
                ax.set_title(f"{p}: {title}"); ax.set_axis_off()
                fig.canvas.draw()
                image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
                image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
                frames.append(image); plt.close()
            gif_local = os.path.join(LOCAL_OUTDIR, f"anim_{p.replace(' ','_')}.gif")
            imageio.mimsave(gif_local, frames, duration=1.0)
            print(f"Saved local: {gif_local}")
            if is_gcs(OUTPUT_DIR_GCS):
                gcs_cp(gif_local, os.path.join(OUTPUT_DIR_GCS, os.path.basename(gif_local)))
        else:
            print(f"Skipping rasters for {p} (files not found)")
else:
    print("Raster section skipped (rasterio not available).")

print("All figures saved locally to:", LOCAL_OUTDIR)
if is_gcs(OUTPUT_DIR_GCS):
    print("Copies pushed to:", OUTPUT_DIR_GCS)

Saved local: /content/figs/stacked_bars_province_stable_gain_loss.png


/tmp/ipython-input-3891718350.py:149: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=CSV_PROV_FIELD, y="ha_prob_change", inner="quartile", palette="Set2", ax=ax, cut=0)


Saved local: /content/figs/violin_change_by_province.png
Saved local: /content/figs/ecdf_change_by_province.png
Saved local: /content/figs/ranked_change_by_adminpost_faceted.png
Unmatched polygons: 214


/usr/local/lib/python3.12/dist-packages/mapclassify/classifiers.py:1767: UserWarning: Not enough unique values in array to form 5 classes. Setting k to 4.
  self.bins = quantile(y, k=k)


Saved local: /content/figs/map_baseline.png


/usr/local/lib/python3.12/dist-packages/mapclassify/classifiers.py:1767: UserWarning: Not enough unique values in array to form 5 classes. Setting k to 4.
  self.bins = quantile(y, k=k)


... (7 outputs left out)

/usr/local/lib/python3.12/dist-packages/mapclassify/classifiers.py:1767: UserWarning: Not enough unique values in array to form 5 classes. Setting k to 1.
  self.bins = quantile(y, k=k)


Saved local: /content/figs/map_relse_change.png
Saved local: /content/figs/hex_end_vs_base.png
Saved local: /content/figs/funnel_se_vs_mean.png


/tmp/ipython-input-3891718350.py:366: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(gdf_valid)
/usr/local/lib/python3.12/dist-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
 There are 3 islands with ids: 12, 116, 367.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 12, ' is an island (no neighbors)')
('WARNING: ', 116, ' is an island (no neighbors)')
('WARNING: ', 367, ' is an island (no neighbors)')
Global Moran's I (change): I=0.670, p=0.0010, z=21.17
Saved local: /content/figs/moran_scatter_change.png
CommandException: No URLs matched: gs://moz-lulc/MOZ_cultivated_change_baseline_prob_2018_2020_Cabo_Delgado.tif
CommandException: 1 file/object could not be transferred.

Raster missing for Cabo Delgado: gs://moz-lulc/MOZ_cultivated_change_baseline_prob_2018_2020_Cabo_Delgado.tif (gsutil cp failed: gs://moz-lulc/MOZ_cultivated_change_baseline_prob_2018_2020_Cabo_Delgado.tif -> /content/rasters/MOZ_cultivated_change_baseline_prob_2018_2020_Cabo_Delgado.tif)
CommandException: No URLs matched: gs://moz-lulc/MOZ_cultivated_change_endline_prob_2022_2024_Cabo_Delgado.tif
CommandException: 1 file/object could not be transferred.

Raster missing for Cabo Delgado: gs://moz-lulc/MOZ_cultivated_change_endline_prob_2022_2024_Cabo_Delgado.tif (g